# Intelligent Error Handling with RAG and IBM watsonx

This notebook demonstrates how to use Retrieval-Augmented Generation (RAG) combined with IBM watsonx to respond more effectively to application errors encountered by users.

## Process Description

1. **Knowledge Base Creation**: A set of historical application errors is documented, each with its corresponding root cause analysis (RCA) and runbook. This serves as the foundation for a retrieval system.

2. **Error Matching and Retrieval**: When a new user error is received, vector similarity is used to identify the most similar past incidents from the knowledge base.

3. **AI-Powered Response Generation**: IBM watsonx is used to generate a helpful, conversational response to the user by combining retrieved RCA and runbook content with natural language generation.

## Why This Matters

This approach enables faster and more user-friendly support by:

* Leveraging existing operational knowledge.
* Standardizing responses with verified RCAs and recovery steps.
* Scaling expert knowledge across teams.


## Depencies

- Python 3.x
- scikit-learn
- ibm-watson-machine-learning

In [40]:
!pip install scikit-learn
!pip install ibm-watson-machine-learning


[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


## Vectorize database

This function converts a list of text documents into a numerical format using TF-IDF (Term Frequency–Inverse Document Frequency), which is commonly used in natural language processing to represent how important a word is in a document relative to a collection of documents.

In [41]:
from sklearn.feature_extraction.text import TfidfVectorizer

def generate_tfidf_representation(documents):
    vectorizer = TfidfVectorizer()
    tfidf_vector = vectorizer.fit_transform(documents)
    return tfidf_vector

### **Function Name**: `search(phrase)`

Given a `phrase` (like an error message), find the most similar document from a predefined `knowledge_base`.

* `phrase`: A string you want to search for — typically a new query or error description.
* `knowledge_base`: (not shown in the code, but expected to be defined globally) — a list of text documents (strings) where each item might be an error description, incident, etc.

---

### **How It Works**:

1. **Combine Input and Knowledge Base**:

   ```python
   texts = [phrase] + knowledge_base
   ```

   Puts the query at the start of the list to ensure it gets vectorized along with the rest.

2. **TF-IDF Vectorization**:

   ```python
   tfidf_matrix = generate_tfidf_representation(texts)
   ```

   Converts all texts (query + documents) into TF-IDF vectors.

3. **Split the Vectors**:

   ```python
   phrase_vector = tfidf_matrix[0]
   document_vectors = tfidf_matrix[1:]
   ```

   * `phrase_vector` = vector for the input phrase
   * `document_vectors` = vectors for all the documents in the knowledge base

4. **Compute Cosine Similarity**:

   ```python
   similarities = cosine_similarity(phrase_vector, document_vectors).flatten()
   ```

   Measures how similar the phrase is to each document. Returns values from 0 (no match) to 1 (perfect match).

5. **Find Best Match**:

   ```python
   closest_index = similarities.argmax()
   ```

   Gets the index of the document with the highest similarity score.

---

### **Output**:

* `closest_index`: The index of the most similar document **in the original `knowledge_base`**.

In [42]:
from sklearn.metrics.pairwise import cosine_similarity
def search( phrase ):
    texts = [phrase] + knowledge_base
    tfidf_matrix = generate_tfidf_representation(texts)
    # Compute cosine similarity between the phrase and each document
    phrase_vector = tfidf_matrix[0]  # The TF-IDF vector for the phrase
    document_vectors = tfidf_matrix[1:]  # The TF-IDF vectors for the documents
    # Calculate cosine similarity
    similarities = cosine_similarity(phrase_vector, document_vectors).flatten()
    # Find the index of the document with the highest similarity
    closest_index = similarities.argmax() 
    return closest_index

###  **Function Name**: `generate(model_in, augmented_prompt_in)`
Send a prompt (`augmented_prompt_in`) to a generative model (`model_in`) and extract the resulting text response.

* `model_in`: An object representing a language model (e.g., IBM watsonx, Hugging Face model, etc.) that has a `.generate()` method.
* `augmented_prompt_in`: A string prompt to pass into the model — typically preprocessed or enriched text for better results.

---

### **How It Works**:

1. **Generate a Response from the Model**:

   ```python
   generated_response = model_in.generate(augmented_prompt_in)
   ```

   Calls the model's `generate` method with the input prompt.

2. **Extract the Generated Text if Available**:

   ```python
   if ("results" in generated_response)
       and (len(generated_response["results"]) > 0)
       and ("generated_text" in generated_response["results"][0]):
       return generated_response["results"][0]["generated_text"]
   ```

   This safely checks if the model's response is valid and contains at least one result with a `generated_text` field.

3. **Handle Failure Case Gracefully**:

   ```python
   else:
       print("The model failed to generate an answer")
       print("\nDebug info:\n" + json.dumps(generated_response, indent=3))
       return ""
   ```

   If something is missing (e.g., no results or no `generated_text`), it prints the full response as debug information and returns an empty string.

---

### **Output**:

* If successful: the generated text from the model.
* If failed: prints debug info and returns an empty string.

In [43]:
import json

def generate( model_in, augmented_prompt_in ):
    
    generated_response = model_in.generate( augmented_prompt_in )
    if ( "results" in generated_response ) \
       and ( len( generated_response["results"] ) > 0 ) \
       and ( "generated_text" in generated_response["results"][0] ):
        return generated_response["results"][0]["generated_text"]
    else:
        print( "The model failed to generate an answer" )
        print( "\nDebug info:\n" + json.dumps( generated_response, indent=3 ) )
        return ""

## Database

1. **Defines a dataset** of typical system or application errors (`documents`) that might occur in real-world infrastructure, production systems, or CI/CD pipelines.
2. **Converts this structured data into plain text entries** (`knowledge_base`) that can be searched, compared, or passed to a language model (e.g., for RAG – Retrieval-Augmented Generation).

#### `documents` list:

Each entry is a **dictionary with 3 keys**:

* `"error"`: A brief description of the issue.
* `"rca"`: A diagnosis of what caused the issue (Root Cause Analysis).
* `"runbook"`: A detailed set of instructions for resolving the issue.


takes each dictionary and **formats it into a string** — merging all 3 parts into a text block that can be:

* Vectorized (e.g., using TF-IDF)
* Indexed and searched
* Passed into a prompt for a language model

Example output:

```
"Error: Cart API Returns 500\nRCA: NullPointerException due to missing user context in request headers.\nRunbook:\n1. Reproduce request locally with Postman.\n2. Fix API to validate user context.\n3. Deploy patch.\n4. Write test to cover missing header case."
```

In [44]:
documents = [
   {
        "error": "High Latency in User Login Service",
        "rca": "Database connection pool was exhausted during traffic spike.",
        "runbook": "1. Scale up the database connection pool.\n2. Restart the login service pods.\n3. Monitor connection usage with Prometheus.\n4. Add alerts if usage > 90%."
    },
    {
        "error": "Payment Service Timeout",
        "rca": "External payment gateway took longer than 30s, exceeding timeout threshold.",
        "runbook": "1. Check third-party gateway status page.\n2. Increase timeout threshold temporarily to 60s.\n3. Notify customer support.\n4. Reprocess failed transactions from the queue."
    },
    {
        "error": "Cart API Returns 500",
        "rca": "NullPointerException due to missing user context in request headers.",
        "runbook": "1. Reproduce request locally with Postman.\n2. Fix API to validate user context.\n3. Deploy patch.\n4. Write test to cover missing header case."
    },
    {
        "error": "Search Function Returns Incomplete Results",
        "rca": "One shard in Elasticsearch cluster was marked as 'unassigned'.",
        "runbook": "1. Run `GET _cluster/health`.\n2. Reallocate the unassigned shard using Kibana.\n3. Add disk space if over 85% full.\n4. Add redundancy checks in pipeline."
    },
    {
        "error": "Service Deployment Fails in CI/CD Pipeline",
        "rca": "Docker image tag mismatch between dev and staging environments.",
        "runbook": "1. Check last successful build tag in Jenkins.\n2. Update staging configuration.\n3. Trigger redeploy.\n4. Implement automated tag sync script."
    },
    {
        "error": "Mobile App Crashes on Launch (Android)",
        "rca": "Incompatible library version introduced in latest build.",
        "runbook": "1. Roll back to previous stable release.\n2. Identify breaking changes in changelog.\n3. Apply patch and retest.\n4. Push hotfix to Play Store."
    },
    {
        "error": "Email Notifications Not Sent",
        "rca": "SMTP relay server was blocked due to credential rotation.",
        "runbook": "1. Generate new credentials in SMTP dashboard.\n2. Update secrets in Kubernetes.\n3. Restart mailer service.\n4. Confirm delivery with test email."
    },
    {
        "error": "High CPU Usage in Recommendation Engine",
        "rca": "Infinite loop caused by corrupt user profile data.",
        "runbook": "1. Analyze logs for user IDs with loop behavior.\n2. Cleanse corrupted user data.\n3. Patch logic to avoid infinite processing.\n4. Add safeguard validation in ingestion pipeline."
    },
    {
        "error": "Analytics Dashboard Not Loading",
        "rca": "Frontend is requesting an API endpoint that was deprecated.",
        "runbook": "1. Roll forward frontend to latest version.\n2. Re-enable deprecated API temporarily.\n3. Notify frontend team to update request URLs.\n4. Document deprecation in shared schema."
    },
    {
        "error": "Frequent Crashes in Chatbot Backend",
        "rca": "Memory leak in session handling logic.",
        "runbook": "1. Deploy memory profiler in staging.\n2. Identify objects not being garbage collected.\n3. Refactor code to properly dispose sessions.\n4. Set up memory usage alerts in Datadog."
    }
]

knowledge_base = [
    f"Error: {doc['error']}\n Context: {doc['rca']}\n Steps to fix it:\n{doc['runbook']}"
    for doc in documents
]

## Prompt as template

The goal is to build a **Retrieval-Augmented Generation (RAG)** style prompt where:

* You provide **context** from a document (like an article, error description, or runbook).
* You ask a **question** about it.
* The model is instructed to answer **only using the context provided**.

This keeps the model grounded and reduces hallucinations.

* The first `%s` is for the **context** (e.g., a paragraph, document, runbook, or RCA).
* The second `%s` is for the **user’s question**.

#### Example (once filled):

```text
Document:
###
Error: Cart API Returns 500
RCA: NullPointerException due to missing user context in request headers.
Runbook:
1. Reproduce request locally with Postman.
2. Fix API to validate user context.
3. Deploy patch.
4. Write test to cover missing header case.
###

Answer the following question using only information from the Document. 
Answer in a complete sentence, with proper capitalization and punctuation. 

Question: What caused the Cart API to return 500?
Answer:
```

This format gives the model a clean and grounded context and a clear task.

---

#### 2. **`augment` Function**

```python
def augment(template_in, context_in, query_in):
    return template_in % (context_in, query_in)
```

This function simply fills the placeholders in the prompt template with:

* `context_in`: the relevant article or document (from your `knowledge_base`)
* `query_in`: the user's question

This returns a full prompt ready to send to `watsonx.ai` or another LLM.

In [45]:
prompt_template = """
Document:
###
%s
###

Answer the following question using only information from the Document.
Answer incluing also the steps to fix it from the Document. 
Answer in a complete sentence, with proper capitalization and punctuation. 

Question: %s
Answer: 
"""

def augment( template_in, context_in, query_in ):
    return template_in % ( context_in, query_in )

## IBM cloud

It's setting up a large language model (in this case, Google's FLAN-T5 XXL) so you can generate text — like answering questions, summarizing, or generating recommendations — from IBM's cloud.

* `model_id`: chooses the specific foundation model (here, Google’s FLAN-T5 XXL).
* `api_key`, `region`, `project_id`: loaded from environment variables to keep secrets safe and reusable.

  * You need to set these in your environment or a `.env` file before running this.

In [46]:
import os
from ibm_watson_machine_learning.foundation_models import Model

gen_parms = { 
    "DECODING_METHOD" : "greedy", 
    "MIN_NEW_TOKENS" : 1, 
    "MAX_NEW_TOKENS" : 50 
}

model_id = "ibm/granite-3-2-8b-instruct"
api_key = os.getenv("IBM_API_KEY")
region = os.getenv("IBM_REGION")
project_id = os.getenv("IBM_PROJECT_ID")

credentials = {
    "apikey": f"{api_key}",
    "url": f"https://{region}.ml.cloud.ibm.com"  # Replace with your region
}

model = Model( model_id, credentials, gen_parms, project_id )

## Final integration

This function `searchAndAnswer(model)` is a **full flow** that takes a user's question, retrieves the most relevant known error from a knowledge base, and then uses an **LLM (like one from IBM watsonx)** to generate a helpful answer based on that.

It combines search, retrieval-augmented generation (RAG), and user interaction in a single step.

* Displays:

  * The original question
  * The generated answer from the model
  * The RCA (Root Cause Analysis) and runbook as reference

This is an interactive **RAG (Retrieval-Augmented Generation)** demo using:

* A **knowledge base** of past error resolutions.
* A **TF-IDF + cosine similarity search** to find similar cases.
* A **Watsonx LLM** to generate a natural-language answer.
* A **prompt template** to control how the model responds.

In [47]:
import re
def searchAndAnswer( model ):
    
    question = input( "Type your question:\n")
    if not re.match( r"\S+", question ):
        print( "No question")
        return
        
    # Retrieve the relevant content
    top_matching_index = search( question )
    if top_matching_index < 0:
        print( "No good answer was found in the knowledge base" )
        return;

    asset_txt = knowledge_base[top_matching_index]
    if top_matching_index >= len(documents)   :
        print( "\nQuestion:\n" + question )
        print( "\nCurrent document: \"" + asset_txt )
        print( "\nYour´re done. Thank!\n")
    else:
        # Augment a prompt with context
        augmented_prompt = augment( prompt_template, asset_txt, question )
        print("-------------------------------augmented prompt---------------------------------------------")
        print(augmented_prompt)
        print("----------------------------------------------------------------------------")
        # Generate output
        output = generate( model, augmented_prompt )
        print( "\nQuestion:\n" + question )
        print( "\nAnswer:\n" + output )

# Android Crash Troubleshooting — Prompt Engineering Demo

The purpose of this test is to simulate how to use Retrieval-Augmented Generation (RAG) to answer user error queries more effectively by leveraging previous incident knowledge. Specifically, this test focuses on the question:

> **"Fix the Android crash"**

Using a set of pre-defined incident documents, the program performs the following steps:

1. **Query Interpretation**: Receives the user's question related to an issue (in this case, an Android app crash).
2. **Contextual Retrieval**: Identifies the most relevant document from the knowledge base based on semantic similarity (using TF-IDF and cosine similarity).
3. **Prompt Augmentation**: Injects the retrieved context into a structured prompt template designed for the selected AI model.
4. **Answer Generation**: Calls the model (e.g., `ibm/granite-3-2-8b-instruct` hosted on IBM watsonx) to generate a natural language response that addresses the user’s question.
5. **Response Display**: Outputs the final answer, along with the source RCA (Root Cause Analysis) and runbook steps from the knowledge base.


## Example Scenario

**Input:**

```text
Fix the android crash
```

**Model Output:**

> The crash was due to an incompatible library version introduced in the latest Android build. Roll back to the previous stable release, identify the breaking changes in the changelog, apply a patch, and republish the app.

In [48]:
searchAndAnswer( model )

-------------------------------augmented prompt---------------------------------------------

Document:
###
Error: Mobile App Crashes on Launch (Android)
 Context: Incompatible library version introduced in latest build.
 Steps to fix it:
1. Roll back to previous stable release.
2. Identify breaking changes in changelog.
3. Apply patch and retest.
4. Push hotfix to Play Store.
###

Answer the following question using only information from the Document.
Answer incluing also the steps to fix it from the Document. 
Answer in a complete sentence, with proper capitalization and punctuation. 

Question: Android crash
Answer: 

----------------------------------------------------------------------------

Question:
Android crash

Answer:

The Android mobile app is crashing on launch due to an incompatible library version introduced in the latest build. To fix this issue, follow these steps: Roll back to the previous stable release, identify the breaking changes in the changelog, apply the nece